# Identity, Governance & Access

In the cloud, the network perimeter is gone. Your application is reachable from anywhere; your control plane is reachable from anywhere. The only thing standing between an attacker and your data is *who they prove themselves to be* and *what you have decided they can do*. That is identity and access control, and in Azure the front door for both is **Microsoft Entra ID**.

Get this layer right — strong identities, least-privilege roles, policy guardrails — and most of cloud security falls into place. Get it wrong, and no amount of firewall configuration will save you.

## Microsoft Entra ID

Entra ID — renamed from Azure Active Directory in 2023 but still the same service — is Microsoft's cloud identity provider. It holds your users, groups, applications, devices, and the policies that govern how they sign in. Every Azure subscription is anchored to exactly one Entra ID tenant, and every authentication into Azure goes through it.

Two things to get straight up front:

- **Entra ID is not a cloud version of on-premises Active Directory.** They share heritage and a name, but the protocols differ. On-prem AD speaks Kerberos and LDAP; Entra ID speaks OAuth 2.0, OpenID Connect, and SAML. You cannot point a Windows domain-joined server at Entra ID for legacy authentication — that is what **Entra Domain Services** (a separate service) was built for.
- **A tenant is the identity boundary, not the subscription.** A single tenant can hold many subscriptions; an organisation might run one tenant for the company and another for a regulated subsidiary. Users in tenant A cannot directly access resources in tenant B without an explicit guest invitation.

The AWS analogue is loose. IAM is per-account; Entra ID is per-organisation and spans every subscription underneath. AWS Identity Center (formerly SSO) is closer to Entra ID's enterprise role, but Entra is the older, deeper product — it also powers Microsoft 365, Dynamics 365, and thousands of third-party SaaS apps.

## Users and groups

Inside a tenant, three principal types matter most:

- **Users** — humans. Each user has a UPN (User Principal Name) like `alice@contoso.com`. They can be cloud-only (created in Entra) or synced from on-prem AD.
- **Groups** — collections of users (or other principals). Two flavours: **security groups** for granting access, and **Microsoft 365 groups** for collaboration (mailbox, Teams, SharePoint). For governance you care about security groups.
- **Service principals** — non-human identities that represent applications. Covered next.

Groups can be **assigned** (you add members manually) or **dynamic** (membership is computed from a rule, e.g. `user.department -eq "Finance"`). Dynamic groups eliminate hand-curation but require an Entra ID P1 licence.

For RBAC at scale, the pattern is always: **assign roles to groups, not individual users**. Even if a group has one member today. When responsibilities shift, you change group membership — not dozens of role assignments scattered across resource groups. A six-month-old subscription where every assignment is per-user is the surest sign of a directory that nobody is steering.

## App registrations, service principals & managed identities

When code (not a human) needs to call Azure — a CI/CD pipeline deploying templates, a function reading a secret, a daemon writing to storage — it needs an identity. Entra ID gives you three options, in order of preference:

- **App registration** is the *definition* of an application: its name, the APIs it can call, and the credentials it can use. It lives in Entra ID. Think of it as the class.
- **Service principal** is the *instance* of that app inside a tenant — the actual identity that role assignments bind to. Think of it as the object. One app registration can have service principals in multiple tenants (multi-tenant apps).
- **Managed identity** is Azure's *fully managed* workload identity. Microsoft creates the underlying service principal for you, rotates the credentials automatically, and ties the lifecycle to the resource. Two flavours:
  - **System-assigned** — created with the resource, dies with it. One-to-one.
  - **User-assigned** — a standalone resource you can attach to many resources. Outlives any one of them.

**Always prefer managed identities** when both sides of the call are Azure. They eliminate the entire credential storage problem — no secrets to stash in Key Vault, no rotation jobs, no leaked client secrets in CI logs. Fall back to a service principal with a **federated credential** (workload identity federation) when calling Azure from GitHub Actions, GitLab, or another OIDC-aware system. Fall back to a client secret only when nothing else works.

The AWS comparison: app registration + service principal ≈ IAM application + role; managed identity ≈ IAM role attached to an EC2 instance via instance profile; workload identity federation ≈ IAM role assumed via OIDC.

## Conditional Access and MFA

Authentication in Entra ID is governed by **Conditional Access (CA) policies** — Microsoft's policy language for "if signal X, require action Y". A policy has three parts:

- **Assignments** — who and what it applies to (users, groups, applications, devices, locations, sign-in risk).
- **Access controls — grant** — what must be true to allow access (MFA, compliant device, hybrid-joined device, approved app).
- **Access controls — session** — what restrictions apply once granted (session lifetime, sign-in frequency, app-enforced restrictions).

Common policies in practice:

- All users must use MFA for the Azure portal.
- Admin roles require **phishing-resistant MFA** (FIDO2 key or Windows Hello).
- Sign-ins from outside trusted countries are blocked or require step-up auth.
- Mobile device access requires the device be marked compliant by Intune.

CA evaluation is **always-on**: every sign-in is evaluated against every applicable policy. Multiple policies combine — a sign-in must satisfy *all* grant controls from *all* applicable policies. That makes ordering irrelevant but emergent behaviour subtle; the "What If" tool in the Entra portal is your friend when policies start interacting.

One historical note: **security defaults** is a free, one-click baseline that requires MFA for all users and blocks legacy authentication. Use it for small tenants without P1 licences. Tenants with Conditional Access licences should disable security defaults and write explicit policies instead.

## External identities — B2B and B2C

Two distinct products, easy to confuse:

- **Entra External ID for B2B collaboration** invites users from *other organisations* into your tenant as **guests**. They keep their home identity (`bob@partner.com`) and authenticate through their own tenant; your tenant grants them access to specific resources. Use this for contractors, vendors, partners. Cross-tenant access settings control which tenants can guest into yours and what they can do once they arrive.
- **Entra External ID for customers** (the rebrand of Azure AD B2C) is for *your application's end users*. Customers sign up directly to a customer-facing tenant that you brand and customise; you write user flows for sign-up, sign-in, password reset, and social login (Google, Facebook, Apple). This is what powers consumer apps where you don't want to use the corporate directory.

The AWS analogue: B2B ≈ IAM Identity Center external accounts; customer B2C ≈ Cognito user pools. The dividing line is **who owns the user account** — B2B leaves it with the partner, B2C creates it in a customer-facing directory you control.

## Hybrid identity — Entra Connect

Enterprises rarely start fresh. Most run on-prem Active Directory with thousands of user accounts, and they need those users to sign in to Azure and Microsoft 365 with the same credentials they use for domain-joined laptops. **Entra Connect** is the synchronisation engine that bridges the two directories.

Three sign-in methods, in order of operational simplicity:

- **Password hash synchronisation (PHS)** — a hash of the user's password hash is synced to Entra ID. Authentication happens entirely in the cloud. No on-prem dependency at sign-in time; survives a datacenter outage. The default and recommended choice.
- **Pass-through authentication (PTA)** — Entra forwards the password to an on-prem agent that validates against AD in real time. Useful when policy forbids any password material leaving the datacenter, but requires the on-prem agent to be highly available.
- **Federation (AD FS)** — Entra redirects sign-in to an on-prem AD FS farm. Most flexible (smart-card auth, complex claims rules) and most operationally expensive. Microsoft's direction is *off* federation for most cases.

Entra Connect is being superseded by **Entra Cloud Sync**, a lighter-weight agent for new deployments that handles multi-forest and filtered scopes more cleanly. If you are setting up hybrid identity in 2026 or later, start with Cloud Sync unless a specific feature forces classic Connect.

In [ ]:
# Explore your Entra ID tenant — assumes `az login` and Directory Reader rights.

# 1. Who am I?
az ad signed-in-user show --query "{name:displayName, upn:userPrincipalName, id:id}" -o table

# 2. First ten users.
az ad user list --query "[].{name:displayName, upn:userPrincipalName}" --top 10 -o table

# 3. First ten security groups.
az ad group list --query "[].{name:displayName, id:id}" --top 10 -o table

# 4. Service principals from app registrations you own.
az ad sp list --show-mine --query "[].{name:displayName, appId:appId}" -o table

# 5. Create a service principal scoped to a resource group (typical CI/CD identity).
# In production, prefer a federated credential over a client secret.
az ad sp create-for-rbac \
  --name "sp-ci-deployer" \
  --role Contributor \
  --scopes /subscriptions/<sub-id>/resourceGroups/rg-app-prod
# Output includes appId, password, tenant — store the password in Key Vault immediately.

## RBAC — the authorization model

Role-based access control (RBAC) is how Azure decides *what* an authenticated principal can do. The model is a triple:

- **Security principal** — who is asking (user, group, service principal, managed identity).
- **Role definition** — what set of actions are allowed (a list of `Microsoft.Compute/virtualMachines/read`, `*/write`, etc.).
- **Scope** — where the actions are allowed.

A **role assignment** binds principal + role to a scope. Scope follows the resource hierarchy:

```
Management Group  ──▶ Subscription ──▶ Resource Group ──▶ Resource
   (broadest)                                                (narrowest)
```

An assignment at a parent scope **inherits down**. Assign Reader at the subscription, and the principal can read every resource group and every resource inside, unless an explicit Deny assignment cuts it off. (Deny assignments are rare — they are created by managed services like Blueprints and Managed Apps, not by you directly.)

Two assignments to the same principal stack **additively**: Reader on the subscription plus Contributor on one resource group means Contributor on that group, Reader everywhere else. Azure has no general "deny overrides allow" rule beyond the rare Deny assignment — effective permission is the *union* of all role grants.

AWS comparison: Azure RBAC ≈ AWS IAM policies attached to roles/users. The shape difference: AWS policies are JSON documents authored per principal; Azure roles are reusable definitions you bind to principals at scopes. Azure leans toward "roles as nouns"; AWS toward "policies as documents".

## Built-in roles and custom roles

Azure ships with several **hundred built-in roles**. Four you will see daily:

- **Owner** — full access including the right to delegate (write `Microsoft.Authorization/*`). Reserve for break-glass.
- **Contributor** — full access *except* delegating roles. The day-to-day "I can build things" role.
- **Reader** — view everything in the scope. No mutation. Safe for auditors, finance, dashboards.
- **User Access Administrator** — manages role assignments but not the underlying resources. Used to delegate the delegation right without granting Owner.

Beyond those four, most roles are **service-scoped**: `Storage Blob Data Contributor`, `Key Vault Secrets User`, `AcrPull`, `Cosmos DB Account Reader Role`, and so on. These grant *data-plane* access — for example, reading blob contents — which is separate from the *management-plane* access that lets you reconfigure the storage account itself. Mixing them up is a common pitfall: a Storage Account Contributor cannot read blobs without also being granted a Storage Blob Data role.

When no built-in role fits cleanly, write a **custom role**. The definition is JSON: a name, a list of allowed `Actions` (control plane), `DataActions` (data plane), `NotActions` to subtract from `Actions`, and an `AssignableScopes` list that limits where the role can be used. Custom roles are tenant-wide objects but only assignable inside the scopes you list. Keep the count low — every custom role is one more thing to audit, one more thing to update when Azure adds a resource provider.

In [ ]:
# Walk a real role assignment end to end.

# 1. Roles whose name contains 'Storage Blob Data'.
az role definition list --query "[?contains(roleName,'Storage Blob Data')].roleName" -o table

# 2. Inspect one role to see its Actions / DataActions.
az role definition list --name "Storage Blob Data Reader" -o json

# 3. Assign Reader to a security group at a resource-group scope.
GROUP_OBJECT_ID=$(az ad group show --group "Finance-Auditors" --query id -o tsv)
az role assignment create \
  --assignee-object-id $GROUP_OBJECT_ID \
  --assignee-principal-type Group \
  --role "Reader" \
  --scope /subscriptions/<sub-id>/resourceGroups/rg-finance-reports

# 4. List effective assignments at a scope, including inherited.
az role assignment list \
  --scope /subscriptions/<sub-id>/resourceGroups/rg-finance-reports \
  --include-inherited -o table

# 5. Define a tight custom role: read all, write nothing, scoped to one RG.
az role definition create --role-definition '{
  "Name": "ReportsReadOnly",
  "Description": "Read-only on resources in the reports RG.",
  "Actions": ["*/read"],
  "NotActions": [],
  "AssignableScopes": ["/subscriptions/<sub-id>/resourceGroups/rg-finance-reports"]
}'

## Azure Policy — declarative guardrails

RBAC controls *who can act*. **Azure Policy** controls *what those actions can produce*. The two compose: RBAC lets a Contributor create virtual machines; Policy ensures the only VMs they create are in approved regions, have a tag for cost-centre, and use managed disks.

A **policy definition** is JSON with three parts:

- **Parameters** — values you fill in at assignment time (e.g. the list of allowed regions).
- **Policy rule** — an `if` condition over resource properties and an `effect`.
- **Metadata** — name, description, category.

Built-in definitions cover most common needs: "Allowed locations", "Allowed virtual machine SKUs", "Require a tag and its value", "Storage accounts should disable public access". You write custom definitions when business rules demand it.

Definitions do nothing until they are **assigned** to a scope (management group, subscription, or resource group), at which point the rule starts evaluating against every resource at and below that scope.

**Initiatives** (also called policy sets) bundle related definitions into one assignment. Microsoft ships large initiatives mapped to standards: Microsoft Cloud Security Benchmark, ISO 27001, NIST 800-53, PCI DSS. Assigning the ISO initiative at the root management group means hundreds of individual definitions start evaluating at once, and the compliance dashboard rolls them up into one score.

The AWS analogue: Azure Policy ≈ AWS Config rules + Service Control Policies merged into one product. SCP-style hard prevention (deny) and Config-style audit (audit, deployIfNotExists) both live in the same engine.

## Policy effects

The **effect** is what the rule does when its `if` matches. The effects you will use most:

- **Audit** — log a non-compliance event but allow the action. Safe to deploy; gives you data without breaking workloads.
- **Deny** — block the request at the ARM layer before the resource provider sees it. The most powerful effect and the easiest to weaponise — start in Audit, validate the false-positive rate, *then* switch to Deny.
- **Append** — adds fields to the resource as it is created (e.g. force `tags.environment` to a default).
- **Modify** — like Append, but can also edit existing resources via a remediation task. Used for tag standardisation, enabling diagnostic settings, and similar fixes.
- **DeployIfNotExists (DINE)** — when a matching resource appears, deploy a *related* resource alongside it. Classic example: when a VM is created, deploy the Log Analytics agent extension. Requires a managed identity on the policy assignment.
- **AuditIfNotExists** — like DINE but only logs; does not deploy.

**Exemptions** let you carve out specific scopes or resources from a policy assignment without removing the assignment itself. They are auditable (each carries a category and an expiry), which keeps "we'll fix it later" exceptions visible.

Order of effects at evaluation time: Disabled, then Append/Modify (transform the request), then Deny (block), then AuditIfNotExists/DINE (post-create reaction). That order is what lets you compose policies that, say, *append* a default tag and then *deny* if the value is still wrong — both effects fire, in that order.

In [ ]:
# Assign a built-in policy and inspect compliance.

# 1. Find a definition by display name.
az policy definition list \
  --query "[?displayName=='Allowed locations'].{name:name, displayName:displayName}" -o table

# 2. Assign it to a subscription, restricting deployments to two regions.
az policy assignment create \
  --name "allowed-locations-prod" \
  --display-name "Restrict to East US / West Europe" \
  --policy "e56962a6-4747-49cd-b67b-bf8b01975c4c" \
  --scope /subscriptions/<sub-id> \
  --params '{"listOfAllowedLocations":{"value":["eastus","westeurope"]}}'

# 3. Inspect non-compliant resources.
az policy state list --subscription <sub-id> \
  --query "[?complianceState=='NonCompliant'].{resource:resourceId, policy:policyDefinitionName}" \
  -o table

# 4. Exempt one resource group while you migrate it.
az policy exemption create \
  --name "rg-legacy-exempt" \
  --policy-assignment "/subscriptions/<sub-id>/providers/Microsoft.Authorization/policyAssignments/allowed-locations-prod" \
  --exemption-category Waiver \
  --resource-group rg-legacy \
  --expires-on 2026-12-31

## Management groups, Blueprints & tags

Management groups (introduced in notebook 01) are the lever for applying policy and RBAC consistently across many subscriptions. The shape most enterprises converge on follows Microsoft's **Cloud Adoption Framework** landing zones:

```
Root MG
 ├── Platform                (shared services: networking, identity, monitoring)
 │    ├── Connectivity sub
 │    ├── Identity sub
 │    └── Management sub
 ├── Landing Zones           (workload subscriptions, organised by stage)
 │    ├── Production
 │    │    ├── Online sub
 │    │    └── Corp sub
 │    └── Non-production
 │         ├── Online sub
 │         └── Corp sub
 ├── Sandbox                 (looser policy, capped spend)
 └── Decommissioned          (read-only, scheduled for deletion)
```

Policies attach high: a "deny public IP" policy lives at the Landing Zones management group and inherits down to every workload subscription. Sandbox gets its own laxer policy set; Decommissioned gets a Deny-all-writes.

**Azure Blueprints** was the original packaged-environment tool — bundles of role assignments, policy assignments, and ARM templates applied as a unit. **Blueprints was deprecated in July 2026** in favour of Azure Deployment Stacks plus Template Specs. Do not start new work on Blueprints; existing customers should migrate.

**Tags** are key-value pairs you attach to resources and resource groups. They are the dimension that cost reporting, billing exports, and most automation scripts pivot on. A common scheme: `environment`, `owner`, `cost-center`, `application`, `data-classification`. Use Policy with the `Modify` effect to enforce required tags — relying on engineers to remember them is how 30% of spend ends up in "untagged".

## Resource locks

Locks are the seatbelt against accidental deletion. Two kinds, both applied at subscription, resource group, or resource scope:

- **CanNotDelete** — read and modify are allowed, but `DELETE` is blocked.
- **ReadOnly** — even modifying is blocked. Be careful — many "read-only" operations are actually `write` calls under the hood, so ReadOnly locks cause more support tickets than CanNotDelete.

Locks are checked by Azure Resource Manager *before* the resource provider sees the request, so they apply uniformly to portal, CLI, SDK, and template deployments. Critically, locks **inherit downward**: a CanNotDelete lock on a resource group protects every resource in it.

Locks intentionally sit *outside* the RBAC role hierarchy. Even an Owner cannot delete a locked resource without first removing the lock — that is the whole point. The only roles with intrinsic lock-management rights are Owner and User Access Administrator (the `Microsoft.Authorization/locks/*` actions are tightly held). Use locks on production resource groups, customer-data storage accounts, peered VNet gateways, and anything else where "oops" is unrecoverable.

## Putting it together

The identity and governance stack composes top to bottom:

```
Entra ID             ──▶ who you are                (authentication)
Conditional Access   ──▶ under what conditions      (sign-in policy)
RBAC                 ──▶ what actions you can take  (authorization)
Azure Policy         ──▶ what shape those actions can produce
Locks                ──▶ what you absolutely cannot destroy
```

Each layer is enforced by ARM at the chokepoint, so the picture is consistent across portal, CLI, SDK, and templates.

When designing a new workload, the order is the same: which **principals** need access, what **role** they need at what **scope**, which **policy guardrails** prevent misconfiguration, and which resources need a **lock** in production. Do those four in order and you have built the spine that every later service in this series will hang off.